# 1. Audit outliers

In [6]:
import pandas as pd
import numpy as np

from ml_challenge.config import RAW_DATA_DIR

train = pd.read_csv(RAW_DATA_DIR / "train.csv")
test = pd.read_csv(RAW_DATA_DIR / "test.csv")

In [7]:
TARGET = "is_fraud"
ID_COL = "id"

num_cols = train.drop(columns=[TARGET, ID_COL]).select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

print(f"Number of numerical features: {len(num_cols)}")
print(num_cols)

Number of numerical features: 13
['amount_usd', 'hours_since_last_txn', 'txn_count_last_24h', 'distance_from_home_km', 'card_age_months', 'customer_age', 'account_balance_usd', 'cvv_retry_count', 'velocity_score', 'time_of_day_hour', 'day_of_week', 'merchant_risk_score', 'prior_disputes']


In [8]:
outlier_summary = []

for col in num_cols:
    q1 = train[col].quantile(0.25)
    q3 = train[col].quantile(0.75)
    
    iqr = q3 - q1
    
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    
    outlier_mask = (
        (train[col] < lower_bound) |
        (train[col] > upper_bound)
    )
    
    outlier_count = outlier_mask.sum()
    outlier_pct = outlier_count / len(train) * 100
    
    outlier_summary.append({
        "feature": col,
        "Q1": q1,
        "Q3": q3,
        "IQR": iqr,
        "lower_bound": lower_bound,
        "upper_bound": upper_bound,
        "outlier_count": outlier_count,
        "outlier_pct": outlier_pct
    })

outlier_summary = pd.DataFrame(outlier_summary)

In [9]:
outlier_summary.sort_values(
    "outlier_pct",
    ascending=False
)

,feature,Q1,Q3,IQR,lower_bound,upper_bound,outlier_count,outlier_pct
12,prior_disputes,0.00,0.00,0.00,0.00,0.00,101564,14.509972
0,amount_usd,15.81,43.11,27.30,-25.14,84.06,70741,10.106435
7,cvv_retry_count,0.00,0.00,0.00,0.00,0.00,57872,8.267901
1,hours_since_last_txn,2.11,9.19,7.08,-8.51,19.81,32816,4.688268
6,account_balance_usd,834.12,2517.02,1682.90,-1690.23,5041.37,29068,4.152809
3,distance_from_home_km,4.43,19.75,15.32,-18.55,42.73,16770,2.395851
11,merchant_risk_score,26.40,42.70,16.30,1.95,67.15,13028,1.861249
8,velocity_score,11.40,26.00,14.60,-10.50,47.90,4276,0.610892
2,txn_count_last_24h,2.00,4.00,2.00,-1.00,7.00,2704,0.386308
4,card_age_months,43.00,51.00,8.00,31.00,63.00,2253,0.321876


In [10]:
outlier_features = outlier_summary[
    outlier_summary["outlier_count"] > 0
]["feature"].tolist()

print("Features with IQR outliers:")
for col in outlier_features:
    print(f"- {col}")

Features with IQR outliers:
- amount_usd
- hours_since_last_txn
- txn_count_last_24h
- distance_from_home_km
- card_age_months
- account_balance_usd
- cvv_retry_count
- velocity_score
- merchant_risk_score
- prior_disputes


In [ ]:
outlier_features = outlier_summary[
    outlier_summary["outlier_count"] > 0
]["feature"].tolist()

print("Features with IQR outliers:")
for col in outlier_features:
    print(f"- {col}")

Features with IQR outliers:
- amount_usd
- hours_since_last_txn
- txn_count_last_24h
- distance_from_home_km
- card_age_months
- account_balance_usd
- cvv_retry_count
- velocity_score
- merchant_risk_score
- prior_disputes


In [11]:
for col in outlier_features:
    print(f"\n===== {col} =====")
    print(train[col].describe())


===== amount_usd =====
count    699960.000000
mean         79.893068
std         205.173538
min           1.000000
25%          15.810000
50%          27.250000
75%          43.110000
max        6872.690000
Name: amount_usd, dtype: float64

===== hours_since_last_txn =====
count    699960.000000
mean          6.633734
std           6.290133
min           0.010000
25%           2.110000
50%           4.730000
75%           9.190000
max          51.980000
Name: hours_since_last_txn, dtype: float64

===== txn_count_last_24h =====
count    699960.000000
mean          2.984092
std           1.498998
min           0.000000
25%           2.000000
50%           3.000000
75%           4.000000
max          11.000000
Name: txn_count_last_24h, dtype: float64

===== distance_from_home_km =====
count    699960.000000
mean         13.587575
std          11.617325
min           0.000000
25%           4.430000
50%          10.220000
75%          19.750000
max          57.930000
Name: distance_from_ho

In [12]:
for col in outlier_features:
    print(f"\n===== {col} =====")
    print(train[col].sort_values(ascending=False).head(10).to_string(index=False))


===== amount_usd =====
6872.69
6872.69
6872.69
6153.69
6078.58
6078.57
6078.29
6058.54
6058.36
6058.19

===== hours_since_last_txn =====
51.98
51.97
51.94
51.94
51.91
51.87
51.87
51.84
51.82
51.78

===== txn_count_last_24h =====
11
11
11
11
11
11
11
11
11
10

===== distance_from_home_km =====
57.93
57.86
57.85
57.67
57.65
57.54
57.54
57.54
57.53
57.53

===== card_age_months =====
73
73
73
72
72
72
72
72
72
72

===== account_balance_usd =====
99701.39
88569.07
88566.00
88566.00
88566.00
88566.00
88566.00
88566.00
88566.00
88566.00

===== cvv_retry_count =====
3
3
3
3
3
3
3
3
3
3

===== velocity_score =====
72.9
72.8
71.9
71.5
71.0
71.0
70.6
70.0
70.0
69.5

===== merchant_risk_score =====
100.0
100.0
100.0
100.0
100.0
100.0
100.0
100.0
100.0
100.0

===== prior_disputes =====
4
4
3
3
3
3
3
3
3
3


## Outlier Audit Decision

IQR was used to identify potential outliers in numerical features.

The analysis identified potential outliers in:
- amount_usd
- hours_since_last_txn
- txn_count_last_24h
- distance_from_home_km
- card_age_months
- account_balance_usd
- cvv_retry_count
- velocity_score
- merchant_risk_score
- prior_disputes

However, IQR outliers were not automatically considered erroneous data.

Several features, especially `prior_disputes` and `cvv_retry_count`, have highly right-skewed distributions with Q1 = Q3 = 0. In these cases, IQR labels positive values as outliers even though they may represent meaningful transaction behavior.

Other extreme values, such as high transaction amounts or account balances, were also not identified as invalid solely based on their magnitude.

Therefore, no observations were removed or clipped based on the IQR audit.

Decision: KEEP all observed values and retain the original numerical features for subsequent preprocessing and feature engineering.

# 2. Data leakage & structural checks

##### 2.1 Outlier Audit

In [25]:
print("Train shape:", train.shape)
print("Test shape :", test.shape)

print("\nTrain columns:")
print(train.columns.tolist())

print("\nTest columns:")
print(test.columns.tolist())

print("\nColumns only in train:")
print(sorted(set(train.columns) - set(test.columns)))

print("\nColumns only in test:")
print(sorted(set(test.columns) - set(train.columns)))

Train shape: (699960, 26)
Test shape : (299984, 25)

Train columns:
['id', 'amount_usd', 'hours_since_last_txn', 'txn_count_last_24h', 'distance_from_home_km', 'card_age_months', 'customer_age', 'account_balance_usd', 'cvv_retry_count', 'velocity_score', 'time_of_day_hour', 'day_of_week', 'merchant_risk_score', 'prior_disputes', 'merchant_category', 'card_type', 'auth_method', 'channel', 'device_type', 'is_foreign_transaction', 'is_new_merchant', 'used_vpn', 'ip_country_mismatch', 'billing_shipping_mismatch', 'is_ai_generated_scam_attempt', 'is_fraud']

Test columns:
['id', 'amount_usd', 'hours_since_last_txn', 'txn_count_last_24h', 'distance_from_home_km', 'card_age_months', 'customer_age', 'account_balance_usd', 'cvv_retry_count', 'velocity_score', 'time_of_day_hour', 'day_of_week', 'merchant_risk_score', 'prior_disputes', 'merchant_category', 'card_type', 'auth_method', 'channel', 'device_type', 'is_foreign_transaction', 'is_new_merchant', 'used_vpn', 'ip_country_mismatch', 'billing

##### 2.2 Train/Test Structure

In [14]:
print("\nTrain dtypes:")
print(train.dtypes)

print("\nTest dtypes:")
print(test.dtypes)


Train dtypes:
id                                int64
amount_usd                      float64
hours_since_last_txn            float64
txn_count_last_24h                int64
distance_from_home_km           float64
card_age_months                   int64
customer_age                      int64
account_balance_usd             float64
cvv_retry_count                   int64
velocity_score                  float64
time_of_day_hour                  int64
day_of_week                       int64
merchant_risk_score             float64
prior_disputes                    int64
merchant_category                   str
card_type                           str
auth_method                         str
channel                             str
device_type                         str
is_foreign_transaction             bool
is_new_merchant                    bool
used_vpn                           bool
ip_country_mismatch                bool
billing_shipping_mismatch          bool
is_ai_generated_scam_atte

##### 2.3 Data Types

In [15]:
print("Train ID unique:", train["id"].is_unique)
print("Test ID unique :", test["id"].is_unique)

Train ID unique: True
Test ID unique : True


##### 2.4 ID Integrity

In [16]:
print("Target in train:", TARGET in train.columns)
print("Target in test :", TARGET in test.columns)

Target in train: True
Target in test : False


##### 2.5 Target and Data Leakage Check

In [17]:
print("\nPotentially suspicious columns:")
for col in train.columns:
    if col != TARGET:
        print(col)


Potentially suspicious columns:
id
amount_usd
hours_since_last_txn
txn_count_last_24h
distance_from_home_km
card_age_months
customer_age
account_balance_usd
cvv_retry_count
velocity_score
time_of_day_hour
day_of_week
merchant_risk_score
prior_disputes
merchant_category
card_type
auth_method
channel
device_type
is_foreign_transaction
is_new_merchant
used_vpn
ip_country_mismatch
billing_shipping_mismatch
is_ai_generated_scam_attempt


##### 2.6 Train/Test ID Overlap

In [19]:
train_test_id_overlap = set(train["id"]).intersection(set(test["id"]))

print("Number of overlapping IDs:", len(train_test_id_overlap))

Number of overlapping IDs: 0


##### 2.7 Categorical Consistency

In [18]:
categorical_cols = train.select_dtypes(include=["object", "string"]).columns.tolist()

for col in categorical_cols:
    train_values = set(train[col].dropna().unique())
    test_values = set(test[col].dropna().unique())

    only_in_train = train_values - test_values
    only_in_test = test_values - train_values

    print(f"\n===== {col} =====")
    print("Train unique:", len(train_values))
    print("Test unique :", len(test_values))
    print("Only in train:", only_in_train)
    print("Only in test :", only_in_test)


===== merchant_category =====
Train unique: 12
Test unique : 12
Only in train: set()
Only in test : set()

===== card_type =====
Train unique: 5
Test unique : 5
Only in train: set()
Only in test : set()

===== auth_method =====
Train unique: 5
Test unique : 5
Only in train: set()
Only in test : set()

===== channel =====
Train unique: 5
Test unique : 5
Only in train: set()
Only in test : set()

===== device_type =====
Train unique: 8
Test unique : 8
Only in train: set()
Only in test : set()


# Data Cleaning Decision Log

##### 1. Outlier Audit

IQR-based analysis was used to identify potential outliers in numerical features.

Potential outliers were identified in:
- `amount_usd`
- `hours_since_last_txn`
- `txn_count_last_24h`
- `distance_from_home_km`
- `card_age_months`
- `account_balance_usd`
- `cvv_retry_count`
- `velocity_score`
- `merchant_risk_score`
- `prior_disputes`

The detected values were inspected using their observed ranges and extreme values.

The IQR method does not by itself indicate that these observations are erroneous. In particular, `prior_disputes` and `cvv_retry_count` have Q1 = Q3 = 0, causing positive values to be classified as IQR outliers even though they may represent meaningful fraud-related behavior.

Other extreme values were also not demonstrated to be invalid based on the available data.

**Decision:** Keep all identified outlier values. No row removal or clipping was performed.

---

##### 2. Train/Test Structure

The training dataset contains 26 columns, while the test dataset contains 25 columns.

The only column present in train but absent from test is `is_fraud`, which is the target variable.

**Decision:** The train/test structure is valid and is kept unchanged.

---

##### 3. Data Types

The data types of corresponding features are consistent between train and test.

Numerical features use numerical data types, categorical features are stored as strings, and Boolean features use Boolean data types.

The target `is_fraud` exists only in the training dataset.

**Decision:** No data type modification was performed during data cleaning.

---

##### 4. ID Integrity

The `id` column is unique within both datasets:

- Train ID unique: `True`
- Test ID unique: `True`

**Decision:** No duplicate IDs were identified within either dataset. The `id` column is retained in the cleaned datasets but will remain excluded from model features.

---

##### 5. Target and Data Leakage Check

The target variable `is_fraud` exists only in the training dataset and is absent from the test dataset.

No feature was identified through the structural inspection as directly containing the target variable.

**Decision:** No feature was removed for data leakage based on the performed checks.

---

##### 6. Train/Test ID Overlap

The number of overlapping IDs between train and test is:

`0`

Therefore, no transaction ID is shared between the training and test datasets.

**Decision:** No ID overlap was detected. The train/test separation is retained.

---

##### 7. Categorical Consistency

The categorical features have the same set of unique values in train and test:

- `merchant_category`: 12 categories
- `card_type`: 5 categories
- `auth_method`: 5 categories
- `channel`: 5 categories
- `device_type`: 8 categories

No category was found exclusively in train or exclusively in test.

**Decision:** No categorical value modification was performed.

---

### Overall Cleaning Decision

Based on the performed outlier, structural, ID, leakage, and categorical consistency checks, no evidence was found that required removing observations or modifying feature values.

Therefore:

- No rows were removed.
- No numerical values were clipped.
- No categorical values were modified.
- No features were removed due to structural leakage.
- The original observations are retained for subsequent preprocessing and feature engineering.

The cleaned datasets will be exported to `data/interim/`.

# Export Cleaned Data

Based on the data cleaning audit, no row-level removal or feature-value modification was required.

The cleaned train and test datasets are therefore exported to `data/interim/` for use in subsequent preprocessing and feature engineering steps.

In [26]:
from ml_challenge.config import INTERIM_DATA_DIR

INTERIM_DATA_DIR.mkdir(parents=True, exist_ok=True)

train_clean_path = INTERIM_DATA_DIR / "train_clean.csv"
test_clean_path = INTERIM_DATA_DIR / "test_clean.csv"

train.to_csv(train_clean_path, index=False)
test.to_csv(test_clean_path, index=False)

print("Cleaned train saved to:", train_clean_path)
print("Cleaned test saved to :", test_clean_path)

Cleaned train saved to: C:\Project\ML-Assignment\data\interim\train_clean.csv
Cleaned test saved to : C:\Project\ML-Assignment\data\interim\test_clean.csv


In [27]:
train_clean = pd.read_csv(train_clean_path)
test_clean = pd.read_csv(test_clean_path)

print("Train clean shape:", train_clean.shape)
print("Test clean shape :", test_clean.shape)

print("\nTrain clean columns:")
print(train_clean.columns.tolist())

print("\nTest clean columns:")
print(test_clean.columns.tolist())

Train clean shape: (699960, 26)
Test clean shape : (299984, 25)

Train clean columns:
['id', 'amount_usd', 'hours_since_last_txn', 'txn_count_last_24h', 'distance_from_home_km', 'card_age_months', 'customer_age', 'account_balance_usd', 'cvv_retry_count', 'velocity_score', 'time_of_day_hour', 'day_of_week', 'merchant_risk_score', 'prior_disputes', 'merchant_category', 'card_type', 'auth_method', 'channel', 'device_type', 'is_foreign_transaction', 'is_new_merchant', 'used_vpn', 'ip_country_mismatch', 'billing_shipping_mismatch', 'is_ai_generated_scam_attempt', 'is_fraud']

Test clean columns:
['id', 'amount_usd', 'hours_since_last_txn', 'txn_count_last_24h', 'distance_from_home_km', 'card_age_months', 'customer_age', 'account_balance_usd', 'cvv_retry_count', 'velocity_score', 'time_of_day_hour', 'day_of_week', 'merchant_risk_score', 'prior_disputes', 'merchant_category', 'card_type', 'auth_method', 'channel', 'device_type', 'is_foreign_transaction', 'is_new_merchant', 'used_vpn', 'ip_cou